In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 32. Week 22 — Kernel ridgeとGaussian process

> smoothなnonlinearityとpredictive uncertaintyを得ても、仮定・計算量・coverageの監査は残る。

## 学習目標

- RBF kernelとkernel ridgeの式を説明できる
- GP posterior meanとvarianceをCholesky solveで計算できる
- length scaleとnoiseをvalidationで選べる
- cubic costのためのfit subsetを明記できる

## 前提知識

- B1のpositive-definite matrixとCholesky
- B5のridgeとtemporal validation
- Gaussian conditional distribution

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 32


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)

assert treasury.quality.accepted
assert np.all(forecast.target_dates > forecast.prediction_dates)
assert np.all(np.isfinite(forecast.features))
crosses_methodology_break = (
    (forecast.prediction_dates < qt.TREASURY_METHOD_BREAK.to_datetime64())
    & (forecast.target_dates >= qt.TREASURY_METHOD_BREAK.to_datetime64())
)
assert not np.any(crosses_methodology_break)

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("rows / forecast rows:", len(rates), len(forecast.regression_target))
print("methodology-crossing targets retained:", int(crosses_methodology_break.sum()))
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
rows / forecast rows: 2750 2728
methodology-crossing targets retained: 0
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. RBF kernelとrepresenter form

$$
k(x,x')=\exp\left(-\frac{\lVert x-x'\rVert_2^2}{2\ell^2}\right),
\qquad
\hat f(x)=k(x,X)(K+\lambda I)^{-1}(y-\bar y)+\bar y.
$$

列尺度はtraining subsetでstandardizeする。explicit inverseは作らずlinear solveを使う。

In [4]:
split = qt.chronological_split(len(forecast.regression_target), gap=1)
features = forecast.features
target = forecast.regression_target

kernel_train = split.train[-500:]
kernel_validation = split.validation
length_scales = [0.5, 1.0, 2.0]
kernel_rows = []
kernel_models = {}
for length_scale in length_scales:
    start = time.perf_counter()
    model = qt.fit_kernel_ridge(
        features[kernel_train],
        target[kernel_train],
        length_scale=length_scale,
        ridge=1.0,
    )
    elapsed = time.perf_counter() - start
    metrics = qt.regression_metrics(
        target[kernel_validation],
        model.predict(features[kernel_validation]),
    )
    kernel_rows.append(
        {
            "length_scale": length_scale,
            "fit_rows": len(kernel_train),
            "validation_rmse_bp": metrics.rmse,
            "fit_seconds": elapsed,
        }
    )
    kernel_models[length_scale] = model
kernel_table = pd.DataFrame(kernel_rows)
display(kernel_table)

,length_scale,fit_rows,validation_rmse_bp,fit_seconds
0,0.5,500,7.186110,0.009865
1,1.0,500,7.187968,0.004780
2,2.0,500,7.215296,0.004877


In [5]:
fig = go.Figure(
    go.Scatter(
        x=kernel_table["length_scale"],
        y=kernel_table["validation_rmse_bp"],
        mode="lines+markers",
    )
)
fig.update_layout(
    title="Kernel length scale selected only on validation",
    xaxis_title="RBF length scale in standardized feature space",
    yaxis_title="Validation RMSE (bp)",
    template="plotly_white",
)
fig.show()

## 2. Gaussian process posterior

GPは同じkernelをcovarianceとして使う。training covarianceを $K+\sigma_n^2 I=LL^\top$ とCholesky分解し、posterior meanとvarianceをsolveで求める。

$$
m_*=\bar y+k_*^\top(K+\sigma_n^2I)^{-1}(y-\bar y),
\qquad
v_*=k(x_*,x_*)+\sigma_n^2-k_*^\top(K+\sigma_n^2I)^{-1}k_*.
$$

$\sigma_n^2$ はtargetのbp²単位で指定する。以下のgridはvalidation RMSEで選び、coverageは同じ
selection期間のdiagnosticなのでfinal保証とは呼ばない。

In [6]:
z90 = 1.6448536269514722
gp_train = split.train[-300:]
gp_rows = []
gp_models = {}
for length_scale in [0.5, 1.0, 2.0]:
    for noise_variance in [10.0, 20.0, 40.0]:
        start = time.perf_counter()
        model = qt.fit_gaussian_process(
            features[gp_train],
            target[gp_train],
            length_scale=length_scale,
            noise_variance=noise_variance,
        )
        elapsed = time.perf_counter() - start
        prediction = model.predict(features[split.validation])
        metrics = qt.regression_metrics(target[split.validation], prediction.mean)
        coverage90 = np.mean(
            (
                target[split.validation]
                >= prediction.mean - z90 * prediction.standard_deviation
            )
            & (
                target[split.validation]
                <= prediction.mean + z90 * prediction.standard_deviation
            )
        )
        setting = (length_scale, noise_variance)
        gp_models[setting] = model
        gp_rows.append(
            {
                "length_scale": length_scale,
                "noise_variance_bp2": noise_variance,
                "validation_rmse_bp": metrics.rmse,
                "validation_coverage90": coverage90,
                "fit_seconds": elapsed,
            }
        )
gp_table = pd.DataFrame(gp_rows)
best_gp_row = gp_table.loc[gp_table["validation_rmse_bp"].idxmin()]
best_gp_setting = (
    float(best_gp_row["length_scale"]),
    float(best_gp_row["noise_variance_bp2"]),
)
gp_model = gp_models[best_gp_setting]
gp_prediction = gp_model.predict(features[split.validation])
display(gp_table)
print("selected GP setting:", best_gp_setting)
print("selection-period empirical 90% coverage:", best_gp_row["validation_coverage90"])
print("fit rows:", len(gp_train))

,length_scale,noise_variance_bp2,validation_rmse_bp,validation_coverage90,fit_seconds
0,0.5,10.0,7.166572,0.754128,0.003600
1,0.5,20.0,7.166572,0.831193,0.001917
2,0.5,40.0,7.166571,0.913761,0.001904
3,1.0,10.0,7.167131,0.754128,0.001856
4,1.0,20.0,7.166938,0.831193,0.002095
5,1.0,40.0,7.166785,0.913761,0.001814
6,2.0,10.0,7.189375,0.752294,0.001954
7,2.0,20.0,7.183126,0.829358,0.002081
8,2.0,40.0,7.177454,0.913761,0.001762


selected GP setting: (0.5, 40.0)
selection-period empirical 90% coverage: 0.9137614678899083
fit rows: 300


In [7]:
display_rows = split.validation[:160]
display_prediction = gp_model.predict(features[display_rows])
display_dates = pd.to_datetime(forecast.prediction_dates[display_rows])
fig = go.Figure()
fig.add_scatter(
    x=np.r_[display_dates, display_dates[::-1]],
    y=np.r_[
        display_prediction.mean + z90 * display_prediction.standard_deviation,
        (display_prediction.mean - z90 * display_prediction.standard_deviation)[::-1],
    ],
    fill="toself",
    line={"color": "rgba(0,0,0,0)"},
    fillcolor="rgba(76,120,168,0.2)",
    name="model-based 90% interval",
)
fig.add_scatter(x=display_dates, y=display_prediction.mean, name="GP mean", mode="lines")
fig.add_scatter(
    x=display_dates,
    y=target[display_rows],
    name="actual",
    mode="lines",
    line={"color": "black", "width": 1},
)
fig.update_layout(
    title="GP uncertainty is model-based and must be coverage-audited",
    xaxis_title="Prediction date",
    yaxis_title="Next-day change (bp)",
    template="plotly_white",
)
fig.show()

## 3. Compute budget

dense kernel solveはfit rowsを $n$ としてmemory $O(n^2)$、time $O(n^3)$である。ridgeが全historyを使う一方、GPは固定300行subsetであることを比較表に残す。これは同じmodel capacityではない。

## 4. 失敗モード

- explicit matrix inverseを作る
- full history GPを暗黙に要求しruntimeを記録しない
- subsetを変えながらscoreだけ比較する
- posterior standard deviationをdistribution-free intervalと呼ぶ
- length scaleをtestで選ぶ

## 5. 段階別演習

### 基礎

1. kernel ridge dual solutionを導出せよ。
2. GP posterior meanとvarianceのsolve順序を書け。

### 標準

3. fit rowsを150、300、500へ変えruntimeを比較せよ。
4. noise varianceとcoverageの関係をvalidationで調べよ。

### 研究

5. inducing-point近似の計算量と誤差を設計せよ。
6. nonstationary kernelが必要な証拠をbreak前後で検討せよ。

## 6. Exit Criteria

- [ ] RBF kernelとkernel ridgeを説明できる
- [ ] Cholesky solveでGP predictionを計算できる
- [ ] fit subsetとruntimeを記録できる
- [ ] uncertaintyとempirical coverageを区別できる
- [ ] length scaleをvalidationだけで選べる

## 7. 出典

- [Gaussian Processes for Machine Learning](https://gaussianprocess.org/gpml/) — kernel、posterior、marginal likelihood、計算量
- [SciPy Cholesky](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.cholesky.html) — positive-definite solveの公式API
- [The Elements of Statistical Learning](https://hastie.su.domains/ElemStatLearn/) — kernel methodとregularization